# Exercise 72 — Mini-project: API → Parquet

## Problem

Pull data from a paginated JSON API, flatten the nested structure, and write to Parquet.

We'll use [JSONPlaceholder](https://jsonplaceholder.typicode.com), a free fake-API for testing. The `/users` endpoint returns 10 user objects with this nested shape:

```json
{
  "id": 1,
  "name": "Leanne Graham",
  "username": "Bret",
  "email": "Sincere@april.biz",
  "address": {"street": "...", "city": "Gwenborough", "zipcode": "...", "geo": {...}},
  "phone": "...",
  "website": "...",
  "company": {"name": "Romaguera-Crona", "catchPhrase": "...", "bs": "..."}
}
```

## Architecture

Split responsibilities so each piece is testable:

1. **`fetch_users()`** — hits the live API. Skipped in offline tests.
2. **`flatten_user(user)`** — pure function, takes one nested user dict, returns a flat row dict. Easily testable with a fixture.
3. **`write_parquet(rows, path)`** — pure function, takes flat rows and writes a Parquet file.
4. **`run_pipeline(...)`** — wires them together.

This is the standard DE shape: an I/O boundary (`fetch`, `write`), a pure transform (`flatten`), and an orchestrator.

## Requirements

```
pip install requests pandas pyarrow
```

If you don't want to install `requests`, the fetch task can be skipped — the other tasks have offline fixtures.

## Fixture for the offline tasks

In [ ]:
FIXTURE_USERS = [
    {
        "id": 1,
        "name": "Leanne Graham",
        "username": "Bret",
        "email": "Sincere@april.biz",
        "address": {"street": "Kulas Light", "city": "Gwenborough", "zipcode": "92998-3874"},
        "phone": "1-770-736-8031 x56442",
        "website": "hildegard.org",
        "company": {"name": "Romaguera-Crona", "catchPhrase": "Multi-layered", "bs": "harness real-time"},
    },
    {
        "id": 2,
        "name": "Ervin Howell",
        "username": "Antonette",
        "email": "Shanna@melissa.tv",
        "address": {"street": "Victor Plains", "city": "Wisokyburgh", "zipcode": "90566-7771"},
        "phone": "010-692-6593 x09125",
        "website": "anastasia.net",
        "company": {"name": "Deckow-Crist", "catchPhrase": "Proactive", "bs": "synergize scalable"},
    },
]
print("fixture ready")


## Task 1 — Flatten one nested user (pure function)

Take one nested user dict and return a flat dict with these exact keys:

```
{
    "id": int,
    "name": str,
    "username": str,
    "email": str,
    "city": str,           # from address.city
    "zipcode": str,        # from address.zipcode
    "company_name": str,   # from company.name
}
```

```python
def flatten_user(user: dict) -> dict:
    ...
```

Don't use any clever recursive flattener — explicitly pick the fields you need. Explicit > magic when the schema is fixed.

In [ ]:
def flatten_user(user: dict) -> dict:
    pass  # TODO

flat = flatten_user(FIXTURE_USERS[0])
assert flat == {
    "id": 1,
    "name": "Leanne Graham",
    "username": "Bret",
    "email": "Sincere@april.biz",
    "city": "Gwenborough",
    "zipcode": "92998-3874",
    "company_name": "Romaguera-Crona",
}
print("ok")


## Task 2 — Write to Parquet (pure function)

Given a list of flat record dicts, write them to a Parquet file. Return the file path.

```python
def write_parquet(rows: list[dict], path: str) -> str:
    ...
```

Approach: `pd.DataFrame(rows).to_parquet(path, index=False)`.

Verify by reading back with `pd.read_parquet` and checking shape + a cell.

In [ ]:
import pandas as pd
from pathlib import Path

def write_parquet(rows: list[dict], path: str) -> str:
    pass  # TODO

flat_rows = [flatten_user(u) for u in FIXTURE_USERS]
path = write_parquet(flat_rows, "users.parquet")
assert path == "users.parquet"
assert Path(path).exists()

df = pd.read_parquet(path)
assert df.shape == (2, 7)
assert df.loc[0, "company_name"] == "Romaguera-Crona"
print("ok")


## Task 3 — End-to-end pipeline (offline)

Write a function that takes a list of users (already fetched), flattens each, writes to Parquet, and returns the row count.

```python
def run_pipeline(users: list[dict], path: str) -> int:
    ...
```

This is what an orchestrator (or a `__main__`) would call. Note that it has no I/O beyond writing — it accepts the data as an argument. This makes it trivially testable in isolation.

In [ ]:
import pandas as pd

def run_pipeline(users: list[dict], path: str) -> int:
    pass  # TODO

n = run_pipeline(FIXTURE_USERS, "users_pipeline.parquet")
assert n == 2
df = pd.read_parquet("users_pipeline.parquet")
assert df.shape == (2, 7)
print("ok")


## Task 4 — Live fetch (optional / network required)

Write `fetch_users()` that does `GET https://jsonplaceholder.typicode.com/users` and returns the parsed JSON list. Use `requests`. Raise on non-200 (`response.raise_for_status()`).

```python
def fetch_users(url: str = "https://jsonplaceholder.typicode.com/users") -> list[dict]:
    ...
```

This cell is **skipped** if you can't or don't want to make a network call — comment it out. The pipeline already works on the fixture data.

In [ ]:
# Uncomment the import and the call below if you want to actually hit the API.
#
# import requests
#
# def fetch_users(url: str = "https://jsonplaceholder.typicode.com/users") -> list[dict]:
#     pass  # TODO
#
# users = fetch_users()
# assert isinstance(users, list)
# assert len(users) == 10
# assert "address" in users[0]
# n = run_pipeline(users, "users_live.parquet")
# assert n == 10
# print("ok")
